# 本地数据探索与重绘

填写数据来源、版本、单位、时间/空间范围、方法、覆盖及缺口。原始数据保持原位置；下方仅处理获准的派生汇总。未运行或无数据时不声称分析完成。


In [ ]:
from pathlib import Path
import subprocess, shlex, re

# 工作目录为本 Notebook 所在目录；只对明确授权的远程任务改为 True。
RUN_REMOTE = False
SSH_ALIAS = ""             # 已配置的 SSH 别名
REMOTE_COMMAND = []        # 如 ["python3", "/remote/project/scripts/summarize.py", ...]
REMOTE_FILE = ""           # 上述命令真实生成的汇总文件的绝对路径
LOCAL_FILE = Path("plot-data/summary.csv")

if RUN_REMOTE:
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", SSH_ALIAS):
        raise ValueError("请设置已配置的 SSH 别名")
    if not REMOTE_COMMAND or not all(isinstance(x, str) and x for x in REMOTE_COMMAND):
        raise ValueError("请设置远程脚本及其参数")
    if not REMOTE_FILE.startswith("/"):
        raise ValueError("请指定远端输出的绝对路径")
    if LOCAL_FILE.exists():
        raise FileExistsError(f"已有本地文件：{LOCAL_FILE}；请换新文件名")
    LOCAL_FILE.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ssh", SSH_ALIAS, shlex.join(REMOTE_COMMAND)], check=True)
    created = False
    try:
        with LOCAL_FILE.open("xb") as output:
            created = True
            subprocess.run(["ssh", SSH_ALIAS, shlex.join(["cat", "--", REMOTE_FILE])], stdout=output, check=True)
        if LOCAL_FILE.stat().st_size == 0:
            raise ValueError("远端结果为空")
    except Exception:
        if created:
            LOCAL_FILE.unlink(missing_ok=True)  # 仅删除本次刚创建的不完整文件
        raise
    print("已保存本地汇总:", LOCAL_FILE)
else:
    print("远程执行未开启；可直接使用本地绘图数据。")


## 本地重新绘图

设置真实数据文件和列名。图的单位、分母、分组和来源应填写在结论中；额外的直方图、时间趋势和空间地图按数据实际适用性添加。


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PLOT_FILE = Path("plot-data/summary.csv")
X_COLUMN = ""  # 实际 x 列
Y_COLUMN = ""  # 实际 y 列
FIGURE = Path("figures/overview.svg")
if not PLOT_FILE.is_file() or not X_COLUMN or not Y_COLUMN:
    print("请提供本地汇总文件及实际列名；尚未生成图。")
else:
    table = pd.read_csv(PLOT_FILE)
    if X_COLUMN not in table or Y_COLUMN not in table:
        raise ValueError("绘图列不在输入表中")
    ax = table.plot(x=X_COLUMN, y=Y_COLUMN, legend=False)
    ax.set_xlabel(X_COLUMN)
    ax.set_ylabel(Y_COLUMN)
    FIGURE.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(FIGURE, bbox_inches="tight")
    plt.show()
    print("图已保存:", FIGURE)


![本地保存的概览图](figures/overview.svg)

## 数据、口径与观察

待填写：来源与版本；数据存放位置；观察单位和字段；时间与空间分辨率；分布；N/A 与真实零值；覆盖分子和分母；异常、限制及尚未完成项。
